# 01 — Document Ingestion and Chunking

## Responsible AI RAG Assistant

This notebook prepares the first technical layer of the project: document ingestion and text chunking.

The goal is to create a clean and reproducible pipeline for loading selected public AI governance sources, extracting text, cleaning the content, and preparing chunks that can later be embedded and stored in a vector database.

## Project Context

This project builds a Retrieval-Augmented Generation assistant for AI governance, GDPR, EU AI Act, and responsible AI knowledge support.

The assistant is designed as an educational and portfolio-ready AI Engineering prototype. It does not provide legal advice.

## Notebook Goals

By the end of this notebook, we want to:

1. Confirm the project folder paths.
2. Define the public source metadata.
3. Prepare local input/output folders.
4. Create helper functions for text cleaning.
5. Create a first chunking strategy.
6. Save processed document chunks for later embedding.

## Important Responsible-Use Note

This project uses public sources only.

Raw downloaded files, generated vector stores, API keys, `.env` files, and private documents should not be uploaded to GitHub.

In [1]:
from pathlib import Path
import pandas as pd
import re
from datetime import date

print("Notebook setup started.")
print(f"Current date: {date.today()}")

Notebook setup started.
Current date: 2026-06-30


In [2]:
# Project paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
DOCS_DIR = PROJECT_ROOT / "docs"
REPORTS_DIR = PROJECT_ROOT / "reports"

paths = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "DATA_DIR": DATA_DIR,
    "RAW_DATA_DIR": RAW_DATA_DIR,
    "PROCESSED_DATA_DIR": PROCESSED_DATA_DIR,
    "DOCS_DIR": DOCS_DIR,
    "REPORTS_DIR": REPORTS_DIR,
}

for name, path in paths.items():
    print(f"{name}: {path}")
    print(f"Exists: {path.exists()}")
    print("-" * 80)

PROJECT_ROOT: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant
Exists: True
--------------------------------------------------------------------------------
DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data
Exists: True
--------------------------------------------------------------------------------
RAW_DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\raw
Exists: True
--------------------------------------------------------------------------------
PROCESSED_DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed
Exists: True
--------------------------------------------------------------------------------
DOCS_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\docs
Exists: True
--------------------------------------------------------------------------------
REPORTS_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\reports
Exists: True
--------------------------------------------------------------------------------


## Source Metadata

The project uses a limited set of public and authoritative sources.

The source metadata below is aligned with `docs/source_register.md`.

For Version 1, the project focuses on:

- EU AI Act overview and official legal text
- GDPR official legal text
- NIST AI Risk Management Framework
- Responsible AI risk-management concepts
- Transparency, human oversight, documentation, and accountability

In [3]:
sources = [
    {
        "source_id": "SRC-001",
        "source_name": "AI Act",
        "publisher": "European Commission",
        "source_type": "Official web page",
        "main_topic": "EU AI Act overview and risk-based approach",
        "url": "https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai",
        "planned_use": "Governance overview and user-facing source explanation",
    },
    {
        "source_id": "SRC-002",
        "source_name": "Regulation (EU) 2024/1689 Artificial Intelligence Act",
        "publisher": "EUR-Lex / European Union",
        "source_type": "Official legal text",
        "main_topic": "AI Act legal provisions, definitions, obligations, and risk categories",
        "url": "https://eur-lex.europa.eu/eli/reg/2024/1689/oj/eng",
        "planned_use": "Grounding source for AI Act questions",
    },
    {
        "source_id": "SRC-003",
        "source_name": "Regulation (EU) 2016/679 General Data Protection Regulation",
        "publisher": "EUR-Lex / European Union",
        "source_type": "Official legal text",
        "main_topic": "GDPR principles, lawful basis, and data protection obligations",
        "url": "https://eur-lex.europa.eu/eli/reg/2016/679/oj/eng",
        "planned_use": "Grounding source for GDPR-related questions",
    },
    {
        "source_id": "SRC-004",
        "source_name": "Artificial Intelligence Risk Management Framework AI RMF 1.0",
        "publisher": "NIST",
        "source_type": "Public framework PDF",
        "main_topic": "AI risk management, trustworthy AI, governance, measurement, mapping, and management",
        "url": "https://nvlpubs.nist.gov/nistpubs/ai/nist.ai.100-1.pdf",
        "planned_use": "Responsible AI and risk-management grounding",
    },
    {
        "source_id": "SRC-005",
        "source_name": "AI Risk Management Framework overview page",
        "publisher": "NIST",
        "source_type": "Official web page",
        "main_topic": "AI risk management framework overview",
        "url": "https://www.nist.gov/itl/ai-risk-management-framework",
        "planned_use": "Supporting source metadata and framework explanation",
    },
]

sources_df = pd.DataFrame(sources)
sources_df

,source_id,source_name,publisher,source_type,main_topic,url,planned_use
0,SRC-001,AI Act,European Commission,Official web page,EU AI Act overview and risk-based approach,https://digital-strategy.ec.europa.eu/en/polic...,Governance overview and user-facing source exp...
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,EUR-Lex / European Union,Official legal text,"AI Act legal provisions, definitions, obligati...",https://eur-lex.europa.eu/eli/reg/2024/1689/oj...,Grounding source for AI Act questions
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,Official legal text,"GDPR principles, lawful basis, and data protec...",https://eur-lex.europa.eu/eli/reg/2016/679/oj/eng,Grounding source for GDPR-related questions
3,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,Public framework PDF,"AI risk management, trustworthy AI, governance...",https://nvlpubs.nist.gov/nistpubs/ai/nist.ai.1...,Responsible AI and risk-management grounding
4,SRC-005,AI Risk Management Framework overview page,NIST,Official web page,AI risk management framework overview,https://www.nist.gov/itl/ai-risk-management-fr...,Supporting source metadata and framework expla...


## Text Cleaning and Chunking Strategy

Before creating embeddings, the source documents must be cleaned and split into smaller text chunks.

Chunking is important because LLMs and vector databases work better when long documents are divided into meaningful, retrievable passages.

For this first version, the project uses a simple word-based chunking strategy:

- clean repeated whitespace
- remove unnecessary line breaks
- split text into chunks of approximately 250 words
- use a 50-word overlap between chunks
- attach source metadata to each chunk

The overlap helps preserve context across chunk boundaries.

In [4]:
def clean_text(text: str) -> str:
    """
    Clean raw document text by normalizing whitespace and removing excessive line breaks.
    
    Parameters
    ----------
    text : str
        Raw input text from a document or web page.
    
    Returns
    -------
    str
        Cleaned text.
    """
    if not isinstance(text, str):
        return ""
    
    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    
    return text


def chunk_text(text: str, chunk_size: int = 250, overlap: int = 50) -> list[str]:
    """
    Split cleaned text into overlapping word-based chunks.
    
    Parameters
    ----------
    text : str
        Cleaned input text.
    chunk_size : int
        Approximate number of words per chunk.
    overlap : int
        Number of overlapping words between consecutive chunks.
    
    Returns
    -------
    list[str]
        List of text chunks.
    """
    if not text:
        return []
    
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")
    
    words = text.split()
    chunks = []
    
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        
        if end >= len(words):
            break
        
        start = end - overlap
    
    return chunks


def create_chunk_records(
    text: str,
    source_id: str,
    source_name: str,
    chunk_size: int = 250,
    overlap: int = 50
) -> list[dict]:
    """
    Clean text, split it into chunks, and attach basic source metadata.
    
    Parameters
    ----------
    text : str
        Raw document text.
    source_id : str
        Unique source identifier.
    source_name : str
        Human-readable source name.
    chunk_size : int
        Number of words per chunk.
    overlap : int
        Number of overlapping words.
    
    Returns
    -------
    list[dict]
        Chunk records with metadata.
    """
    cleaned = clean_text(text)
    chunks = chunk_text(cleaned, chunk_size=chunk_size, overlap=overlap)
    
    records = []
    for idx, chunk in enumerate(chunks, start=1):
        records.append(
            {
                "source_id": source_id,
                "source_name": source_name,
                "chunk_id": f"{source_id}_CHUNK_{idx:03d}",
                "chunk_index": idx,
                "chunk_text": chunk,
                "word_count": len(chunk.split()),
            }
        )
    
    return records


print("Text cleaning and chunking helper functions are ready.")

Text cleaning and chunking helper functions are ready.


In [5]:
sample_text = """
Responsible AI systems should be designed, developed, deployed, and monitored in ways that support transparency, 
accountability, fairness, human oversight, privacy, security, and robustness.

For AI governance, organizations should identify potential risks, document design decisions, evaluate system behavior, 
monitor performance over time, and ensure that appropriate human oversight is available.

When AI systems process personal data, privacy and data protection principles such as lawfulness, fairness, transparency, 
purpose limitation, data minimisation, accuracy, storage limitation, integrity, confidentiality, and accountability may be relevant.

This project does not provide legal advice. It is designed as a knowledge-support prototype that helps users explore public 
AI governance documents and identify concepts that may require further review by qualified legal, privacy, or compliance professionals.
"""

sample_records = create_chunk_records(
    text=sample_text,
    source_id="TEST-001",
    source_name="Sample Responsible AI Text",
    chunk_size=60,
    overlap=15
)

sample_chunks_df = pd.DataFrame(sample_records)
sample_chunks_df

,source_id,source_name,chunk_id,chunk_index,chunk_text,word_count
0,TEST-001,Sample Responsible AI Text,TEST-001_CHUNK_001,1,"Responsible AI systems should be designed, dev...",60
1,TEST-001,Sample Responsible AI Text,TEST-001_CHUNK_002,2,human oversight is available. When AI systems ...,60
2,TEST-001,Sample Responsible AI Text,TEST-001_CHUNK_003,3,a knowledge-support prototype that helps users...,26


## Save First Processed Chunk Table

Before processing real source documents, we first save the sample chunk table locally.

This confirms that the notebook can write processed chunk files to the project structure.

The saved file is only a local development artifact and should not be uploaded to GitHub.

In [6]:
# Validate sample chunk output

required_columns = [
    "source_id",
    "source_name",
    "chunk_id",
    "chunk_index",
    "chunk_text",
    "word_count",
]

missing_columns = [col for col in required_columns if col not in sample_chunks_df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if sample_chunks_df.empty:
    raise ValueError("sample_chunks_df is empty.")

if not sample_chunks_df["chunk_id"].is_unique:
    raise ValueError("chunk_id values are not unique.")

print("Sample chunk validation passed.")
print(f"Number of chunks: {len(sample_chunks_df)}")
print(f"Average words per chunk: {sample_chunks_df['word_count'].mean():.1f}")
print(f"Minimum words per chunk: {sample_chunks_df['word_count'].min()}")
print(f"Maximum words per chunk: {sample_chunks_df['word_count'].max()}")

Sample chunk validation passed.
Number of chunks: 3
Average words per chunk: 48.7
Minimum words per chunk: 26
Maximum words per chunk: 60


In [7]:
# Save sample processed chunks locally

sample_output_path = PROCESSED_DATA_DIR / "sample_responsible_ai_chunks.csv"

sample_chunks_df.to_csv(sample_output_path, index=False, encoding="utf-8")

print(f"Saved sample chunks to: {sample_output_path}")
print(f"File exists: {sample_output_path.exists()}")
print(f"File size in bytes: {sample_output_path.stat().st_size}")

Saved sample chunks to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\sample_responsible_ai_chunks.csv
File exists: True
File size in bytes: 1390


In [8]:
# Read saved file back to confirm reproducibility

loaded_sample_chunks_df = pd.read_csv(sample_output_path)

print(f"Loaded rows: {len(loaded_sample_chunks_df)}")
print(f"Loaded columns: {list(loaded_sample_chunks_df.columns)}")

loaded_sample_chunks_df.head()

Loaded rows: 3
Loaded columns: ['source_id', 'source_name', 'chunk_id', 'chunk_index', 'chunk_text', 'word_count']


,source_id,source_name,chunk_id,chunk_index,chunk_text,word_count
0,TEST-001,Sample Responsible AI Text,TEST-001_CHUNK_001,1,"Responsible AI systems should be designed, dev...",60
1,TEST-001,Sample Responsible AI Text,TEST-001_CHUNK_002,2,human oversight is available. When AI systems ...,60
2,TEST-001,Sample Responsible AI Text,TEST-001_CHUNK_003,3,a knowledge-support prototype that helps users...,26


## Source Inventory and Ingestion Planning

Before collecting real public documents, the project needs a clear source inventory.

The source inventory tracks:

- source ID
- source name
- publisher
- source type
- URL
- planned local raw filename
- collection status
- processing status

This helps keep the RAG pipeline reproducible and prevents confusion between raw files, processed files, and generated vector-store artifacts.

At this stage, the real documents have not been downloaded or processed yet.

In [9]:
# Create source inventory with planned local filenames

source_inventory_df = sources_df.copy()

source_inventory_df["planned_raw_filename"] = [
    "src_001_european_commission_ai_act_overview.txt",
    "src_002_eurlex_ai_act_regulation_2024_1689.txt",
    "src_003_eurlex_gdpr_regulation_2016_679.txt",
    "src_004_nist_ai_rmf_1_0.pdf",
    "src_005_nist_ai_rmf_overview.txt",
]

source_inventory_df["planned_raw_path"] = source_inventory_df["planned_raw_filename"].apply(
    lambda filename: str(RAW_DATA_DIR / filename)
)

source_inventory_df["collection_status"] = "not_collected_yet"
source_inventory_df["processing_status"] = "not_processed_yet"
source_inventory_df["include_in_version_1"] = True

source_inventory_df[
    [
        "source_id",
        "source_name",
        "publisher",
        "source_type",
        "planned_raw_filename",
        "collection_status",
        "processing_status",
        "include_in_version_1",
    ]
]

,source_id,source_name,publisher,source_type,planned_raw_filename,collection_status,processing_status,include_in_version_1
0,SRC-001,AI Act,European Commission,Official web page,src_001_european_commission_ai_act_overview.txt,not_collected_yet,not_processed_yet,True
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,EUR-Lex / European Union,Official legal text,src_002_eurlex_ai_act_regulation_2024_1689.txt,not_collected_yet,not_processed_yet,True
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,Official legal text,src_003_eurlex_gdpr_regulation_2016_679.txt,not_collected_yet,not_processed_yet,True
3,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,Public framework PDF,src_004_nist_ai_rmf_1_0.pdf,not_collected_yet,not_processed_yet,True
4,SRC-005,AI Risk Management Framework overview page,NIST,Official web page,src_005_nist_ai_rmf_overview.txt,not_collected_yet,not_processed_yet,True


In [10]:
# Save source inventory locally

source_inventory_path = PROCESSED_DATA_DIR / "source_inventory.csv"

source_inventory_df.to_csv(source_inventory_path, index=False, encoding="utf-8")

print(f"Saved source inventory to: {source_inventory_path}")
print(f"File exists: {source_inventory_path.exists()}")
print(f"Number of sources: {len(source_inventory_df)}")

Saved source inventory to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\source_inventory.csv
File exists: True
Number of sources: 5


In [11]:
# Check whether planned raw source files already exist locally

raw_file_status = []

for _, row in source_inventory_df.iterrows():
    raw_path = Path(row["planned_raw_path"])
    raw_file_status.append(
        {
            "source_id": row["source_id"],
            "planned_raw_filename": row["planned_raw_filename"],
            "raw_file_exists": raw_path.exists(),
            "raw_path": raw_path,
        }
    )

raw_file_status_df = pd.DataFrame(raw_file_status)
raw_file_status_df

,source_id,planned_raw_filename,raw_file_exists,raw_path
0,SRC-001,src_001_european_commission_ai_act_overview.txt,False,c:\Users\mdadg\Desktop\responsible-ai-rag-assi...
1,SRC-002,src_002_eurlex_ai_act_regulation_2024_1689.txt,False,c:\Users\mdadg\Desktop\responsible-ai-rag-assi...
2,SRC-003,src_003_eurlex_gdpr_regulation_2016_679.txt,False,c:\Users\mdadg\Desktop\responsible-ai-rag-assi...
3,SRC-004,src_004_nist_ai_rmf_1_0.pdf,False,c:\Users\mdadg\Desktop\responsible-ai-rag-assi...
4,SRC-005,src_005_nist_ai_rmf_overview.txt,False,c:\Users\mdadg\Desktop\responsible-ai-rag-assi...


## Document Loading Functions

The next step is to prepare reusable document-loading functions.

The project will use two local source formats:

- `.txt` files for extracted web pages or manually saved public text
- `.pdf` files for public PDF documents such as the NIST AI RMF

The loader functions should:

1. Check whether the file exists.
2. Identify the file type.
3. Load text from `.txt` files.
4. Extract text from `.pdf` files.
5. Return useful metadata such as character count, word count, and page count.

At this stage, the real source files have not been collected yet. The functions are tested using a small local sample text file.

In [12]:
from pypdf import PdfReader


def load_txt_file(file_path: Path, encoding: str = "utf-8") -> str:
    """
    Load text from a local .txt file.
    
    Parameters
    ----------
    file_path : Path
        Path to the text file.
    encoding : str
        File encoding. Default is utf-8.
    
    Returns
    -------
    str
        Loaded text content.
    """
    file_path = Path(file_path)
    
    if not file_path.exists():
        raise FileNotFoundError(f"Text file not found: {file_path}")
    
    if file_path.suffix.lower() != ".txt":
        raise ValueError(f"Expected a .txt file, but received: {file_path.suffix}")
    
    return file_path.read_text(encoding=encoding, errors="replace")


def load_pdf_file(file_path: Path) -> dict:
    """
    Extract text from a local .pdf file.
    
    Parameters
    ----------
    file_path : Path
        Path to the PDF file.
    
    Returns
    -------
    dict
        Dictionary containing extracted text and page-level metadata.
    """
    file_path = Path(file_path)
    
    if not file_path.exists():
        raise FileNotFoundError(f"PDF file not found: {file_path}")
    
    if file_path.suffix.lower() != ".pdf":
        raise ValueError(f"Expected a .pdf file, but received: {file_path.suffix}")
    
    reader = PdfReader(str(file_path))
    page_texts = []
    
    for page_index, page in enumerate(reader.pages, start=1):
        extracted_text = page.extract_text() or ""
        page_texts.append(
            {
                "page_number": page_index,
                "text": extracted_text,
                "character_count": len(extracted_text),
                "word_count": len(extracted_text.split()),
            }
        )
    
    full_text = "\n\n".join(page_record["text"] for page_record in page_texts)
    
    return {
        "text": full_text,
        "page_count": len(reader.pages),
        "page_texts": page_texts,
    }


def load_document_file(file_path: Path) -> dict:
    """
    Load a supported document file and return text plus basic metadata.
    
    Supported formats:
    - .txt
    - .pdf
    
    Parameters
    ----------
    file_path : Path
        Path to the local source file.
    
    Returns
    -------
    dict
        Loaded document text and metadata.
    """
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()
    
    if suffix == ".txt":
        text = load_txt_file(file_path)
        page_count = None
    
    elif suffix == ".pdf":
        pdf_result = load_pdf_file(file_path)
        text = pdf_result["text"]
        page_count = pdf_result["page_count"]
    
    else:
        raise ValueError(f"Unsupported file type: {suffix}")
    
    cleaned_preview = clean_text(text)[:300]
    
    return {
        "file_path": str(file_path),
        "file_name": file_path.name,
        "file_type": suffix,
        "text": text,
        "character_count": len(text),
        "word_count": len(text.split()),
        "page_count": page_count,
        "preview": cleaned_preview,
    }


print("Document loading helper functions are ready.")

Document loading helper functions are ready.


In [13]:
# Create and test a small local sample text file

loader_test_path = PROCESSED_DATA_DIR / "loader_test_sample.txt"

loader_test_text = """
Responsible AI governance requires clear documentation, risk identification, transparency, accountability,
human oversight, and continuous monitoring.

This sample file is used only to test the local document-loading functions before collecting real public source documents.
"""

loader_test_path.write_text(loader_test_text, encoding="utf-8")

loaded_test_document = load_document_file(loader_test_path)

print(f"Loaded file: {loaded_test_document['file_name']}")
print(f"File type: {loaded_test_document['file_type']}")
print(f"Character count: {loaded_test_document['character_count']}")
print(f"Word count: {loaded_test_document['word_count']}")
print(f"Page count: {loaded_test_document['page_count']}")
print()
print("Preview:")
print(loaded_test_document["preview"])

Loaded file: loader_test_sample.txt
File type: .txt
Character count: 276
Word count: 33
Page count: None

Preview:
Responsible AI governance requires clear documentation, risk identification, transparency, accountability, human oversight, and continuous monitoring. This sample file is used only to test the local document-loading functions before collecting real public source documents.


In [14]:
# Check loading readiness for planned raw source files

loader_readiness_records = []

for _, row in source_inventory_df.iterrows():
    raw_path = Path(row["planned_raw_path"])
    
    if raw_path.exists():
        try:
            loaded_doc = load_document_file(raw_path)
            loader_status = "loaded_successfully"
            word_count = loaded_doc["word_count"]
            character_count = loaded_doc["character_count"]
        except Exception as error:
            loader_status = f"loading_error: {error}"
            word_count = None
            character_count = None
    else:
        loader_status = "raw_file_not_available_yet"
        word_count = None
        character_count = None
    
    loader_readiness_records.append(
        {
            "source_id": row["source_id"],
            "source_name": row["source_name"],
            "planned_raw_filename": row["planned_raw_filename"],
            "raw_file_exists": raw_path.exists(),
            "loader_status": loader_status,
            "word_count": word_count,
            "character_count": character_count,
        }
    )

loader_readiness_df = pd.DataFrame(loader_readiness_records)
loader_readiness_df

,source_id,source_name,planned_raw_filename,raw_file_exists,loader_status,word_count,character_count
0,SRC-001,AI Act,src_001_european_commission_ai_act_overview.txt,False,raw_file_not_available_yet,None,None
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,src_002_eurlex_ai_act_regulation_2024_1689.txt,False,raw_file_not_available_yet,None,None
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,src_003_eurlex_gdpr_regulation_2016_679.txt,False,raw_file_not_available_yet,None,None
3,SRC-004,Artificial Intelligence Risk Management Framew...,src_004_nist_ai_rmf_1_0.pdf,False,raw_file_not_available_yet,None,None
4,SRC-005,AI Risk Management Framework overview page,src_005_nist_ai_rmf_overview.txt,False,raw_file_not_available_yet,None,None


## Collect First Real Public Source

The first real source collected for the project is the European Commission AI Act overview page.

This page is used because it is an official public source that explains the AI Act, its risk-based approach, transparency obligations, high-risk AI system concepts, governance, and implementation timeline.

The raw extracted text will be saved locally in `data/raw/`.

The raw source file should not be uploaded to GitHub.

In [15]:
import requests
from bs4 import BeautifulSoup


def fetch_web_page_text(url: str) -> dict:
    """
    Fetch a public web page and extract readable text from headings, paragraphs, and list items.
    
    Parameters
    ----------
    url : str
        Public web page URL.
    
    Returns
    -------
    dict
        Extracted title, text, and basic metadata.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(compatible; ResponsibleAIRAGAssistant/1.0; educational portfolio project)"
        )
    }
    
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.text, "html.parser")
    
    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()
    
    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    
    main_content = soup.find("main") or soup.find("article") or soup.body or soup
    
    text_elements = []
    for element in main_content.find_all(["h1", "h2", "h3", "p", "li"]):
        text = element.get_text(" ", strip=True)
        if text:
            text_elements.append(text)
    
    extracted_text = "\n".join(text_elements)
    cleaned_extracted_text = clean_text(extracted_text)
    
    return {
        "url": url,
        "title": title,
        "status_code": response.status_code,
        "text": extracted_text,
        "cleaned_text": cleaned_extracted_text,
        "character_count": len(extracted_text),
        "word_count": len(extracted_text.split()),
    }


src_001 = source_inventory_df.loc[source_inventory_df["source_id"] == "SRC-001"].iloc[0]

src_001_url = src_001["url"]
src_001_output_path = Path(src_001["planned_raw_path"])

src_001_result = fetch_web_page_text(src_001_url)

src_001_output_path.write_text(src_001_result["text"], encoding="utf-8")

print(f"Fetched URL: {src_001_result['url']}")
print(f"Page title: {src_001_result['title']}")
print(f"Status code: {src_001_result['status_code']}")
print(f"Word count: {src_001_result['word_count']}")
print(f"Character count: {src_001_result['character_count']}")
print(f"Saved to: {src_001_output_path}")
print(f"File exists: {src_001_output_path.exists()}")
print()
print("Preview:")
print(src_001_result["cleaned_text"][:1000])

Fetched URL: https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai
Page title: AI Act | Shaping Europe’s digital future
Status code: 200
Word count: 2229
Character count: 14464
Saved to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\raw\src_001_european_commission_ai_act_overview.txt
File exists: True

Preview:
The AI Act is the first-ever legal framework on AI, which addresses the risks of AI and positions Europe to play a leading role globally. The AI Act (Regulation (EU) 2024/1689 laying down harmonised rules on artificial intelligence) is the first-ever comprehensive legal framework on AI worldwide. The aim of the rules is to foster trustworthy AI in Europe. For any questions on the AI Act , check out the AI Act Single Information platform . The AI Act sets out a risk-based rules for AI developers and deployers regarding specific uses of AI. The AI Act is part of a wider package of policy measures to support the development of trustworthy AI, which al

In [16]:
# Validate the saved SRC-001 raw text file

src_001_loaded = load_document_file(src_001_output_path)

print(f"Loaded file: {src_001_loaded['file_name']}")
print(f"File type: {src_001_loaded['file_type']}")
print(f"Character count: {src_001_loaded['character_count']}")
print(f"Word count: {src_001_loaded['word_count']}")
print(f"Page count: {src_001_loaded['page_count']}")
print()
print("Preview:")
print(src_001_loaded["preview"])

Loaded file: src_001_european_commission_ai_act_overview.txt
File type: .txt
Character count: 14464
Word count: 2229
Page count: None

Preview:
The AI Act is the first-ever legal framework on AI, which addresses the risks of AI and positions Europe to play a leading role globally. The AI Act (Regulation (EU) 2024/1689 laying down harmonised rules on artificial intelligence) is the first-ever comprehensive legal framework on AI worldwide. Th


In [17]:
# Update source inventory after collecting SRC-001

source_inventory_df.loc[
    source_inventory_df["source_id"] == "SRC-001",
    "collection_status"
] = "collected"

source_inventory_df.loc[
    source_inventory_df["source_id"] == "SRC-001",
    "processing_status"
] = "raw_text_saved"

source_inventory_df.loc[
    source_inventory_df["source_id"] == "SRC-001",
    "raw_file_exists"
] = True

source_inventory_df.loc[
    source_inventory_df["source_id"] == "SRC-001",
    "raw_word_count"
] = src_001_loaded["word_count"]

source_inventory_df.loc[
    source_inventory_df["source_id"] == "SRC-001",
    "raw_character_count"
] = src_001_loaded["character_count"]

source_inventory_df.to_csv(source_inventory_path, index=False, encoding="utf-8")

source_inventory_df[
    [
        "source_id",
        "source_name",
        "planned_raw_filename",
        "collection_status",
        "processing_status",
        "raw_file_exists",
        "raw_word_count",
        "raw_character_count",
    ]
]

,source_id,source_name,planned_raw_filename,collection_status,processing_status,raw_file_exists,raw_word_count,raw_character_count
0,SRC-001,AI Act,src_001_european_commission_ai_act_overview.txt,collected,raw_text_saved,True,2229.0,14464.0
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,src_002_eurlex_ai_act_regulation_2024_1689.txt,not_collected_yet,not_processed_yet,NaN,NaN,NaN
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,src_003_eurlex_gdpr_regulation_2016_679.txt,not_collected_yet,not_processed_yet,NaN,NaN,NaN
3,SRC-004,Artificial Intelligence Risk Management Framew...,src_004_nist_ai_rmf_1_0.pdf,not_collected_yet,not_processed_yet,NaN,NaN,NaN
4,SRC-005,AI Risk Management Framework overview page,src_005_nist_ai_rmf_overview.txt,not_collected_yet,not_processed_yet,NaN,NaN,NaN


In [18]:
# Re-check raw file status after collecting SRC-001

updated_raw_file_status = []

for _, row in source_inventory_df.iterrows():
    raw_path = Path(row["planned_raw_path"])
    updated_raw_file_status.append(
        {
            "source_id": row["source_id"],
            "planned_raw_filename": row["planned_raw_filename"],
            "raw_file_exists": raw_path.exists(),
            "file_size_bytes": raw_path.stat().st_size if raw_path.exists() else None,
        }
    )

updated_raw_file_status_df = pd.DataFrame(updated_raw_file_status)
updated_raw_file_status_df

,source_id,planned_raw_filename,raw_file_exists,file_size_bytes
0,SRC-001,src_001_european_commission_ai_act_overview.txt,True,14612.0
1,SRC-002,src_002_eurlex_ai_act_regulation_2024_1689.txt,False,NaN
2,SRC-003,src_003_eurlex_gdpr_regulation_2016_679.txt,False,NaN
3,SRC-004,src_004_nist_ai_rmf_1_0.pdf,False,NaN
4,SRC-005,src_005_nist_ai_rmf_overview.txt,False,NaN


## Chunk First Real Source

Now that the first real public source has been collected, the next step is to clean and chunk the text.

The source is:

`SRC-001 — European Commission AI Act overview page`

The chunking strategy uses:

- approximately 250 words per chunk
- 50-word overlap between chunks
- source metadata attached to every chunk

The processed chunk table will be saved locally in `data/processed/`.

In [19]:
# Create chunks for SRC-001

src_001_metadata = source_inventory_df.loc[
    source_inventory_df["source_id"] == "SRC-001"
].iloc[0]

src_001_text = src_001_loaded["text"]

src_001_chunk_records = create_chunk_records(
    text=src_001_text,
    source_id=src_001_metadata["source_id"],
    source_name=src_001_metadata["source_name"],
    chunk_size=250,
    overlap=50,
)

src_001_chunks_df = pd.DataFrame(src_001_chunk_records)

# Add richer metadata for future retrieval and citation display
src_001_chunks_df["publisher"] = src_001_metadata["publisher"]
src_001_chunks_df["source_type"] = src_001_metadata["source_type"]
src_001_chunks_df["url"] = src_001_metadata["url"]
src_001_chunks_df["planned_raw_filename"] = src_001_metadata["planned_raw_filename"]

# Reorder columns
src_001_chunks_df = src_001_chunks_df[
    [
        "source_id",
        "source_name",
        "publisher",
        "source_type",
        "url",
        "planned_raw_filename",
        "chunk_id",
        "chunk_index",
        "chunk_text",
        "word_count",
    ]
]

print(f"Number of chunks created: {len(src_001_chunks_df)}")
print(f"Average words per chunk: {src_001_chunks_df['word_count'].mean():.1f}")
print(f"Minimum words per chunk: {src_001_chunks_df['word_count'].min()}")
print(f"Maximum words per chunk: {src_001_chunks_df['word_count'].max()}")

src_001_chunks_df.head()

Number of chunks created: 11
Average words per chunk: 248.1
Minimum words per chunk: 229
Maximum words per chunk: 250


,source_id,source_name,publisher,source_type,url,planned_raw_filename,chunk_id,chunk_index,chunk_text,word_count
0,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_001,1,The AI Act is the first-ever legal framework o...,250
1,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_002,2,"of the AI Act ahead of time. In parallel, the ...",250
2,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_003,3,"prohibits eight practices , namely: harmful AI...",250
3,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_004,4,professional life (e.g. scoring of exams) AI-b...,250
4,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_005,5,"high level of robustness, cybersecurity and ac...",250


In [20]:
# Inspect first and last chunk for quality

first_chunk = src_001_chunks_df.iloc[0]
last_chunk = src_001_chunks_df.iloc[-1]

print("FIRST CHUNK")
print("-" * 80)
print(f"Chunk ID: {first_chunk['chunk_id']}")
print(f"Word count: {first_chunk['word_count']}")
print()
print(first_chunk["chunk_text"][:1200])

print("\n\nLAST CHUNK")
print("-" * 80)
print(f"Chunk ID: {last_chunk['chunk_id']}")
print(f"Word count: {last_chunk['word_count']}")
print()
print(last_chunk["chunk_text"][:1200])

FIRST CHUNK
--------------------------------------------------------------------------------
Chunk ID: SRC-001_CHUNK_001
Word count: 250

The AI Act is the first-ever legal framework on AI, which addresses the risks of AI and positions Europe to play a leading role globally. The AI Act (Regulation (EU) 2024/1689 laying down harmonised rules on artificial intelligence) is the first-ever comprehensive legal framework on AI worldwide. The aim of the rules is to foster trustworthy AI in Europe. For any questions on the AI Act , check out the AI Act Single Information platform . The AI Act sets out a risk-based rules for AI developers and deployers regarding specific uses of AI. The AI Act is part of a wider package of policy measures to support the development of trustworthy AI, which also includes the AI Continent Action Plan , the AI Innovation Package and the launch of AI Factories . Together, these measures guarantee safety, fundamental rights and human-centric AI, and strengthen uptak

In [21]:
# Validate SRC-001 chunk table

required_chunk_columns = [
    "source_id",
    "source_name",
    "publisher",
    "source_type",
    "url",
    "planned_raw_filename",
    "chunk_id",
    "chunk_index",
    "chunk_text",
    "word_count",
]

missing_chunk_columns = [
    col for col in required_chunk_columns 
    if col not in src_001_chunks_df.columns
]

if missing_chunk_columns:
    raise ValueError(f"Missing required columns: {missing_chunk_columns}")

if src_001_chunks_df.empty:
    raise ValueError("src_001_chunks_df is empty.")

if not src_001_chunks_df["chunk_id"].is_unique:
    raise ValueError("chunk_id values are not unique.")

if src_001_chunks_df["chunk_text"].isna().any():
    raise ValueError("Some chunks have missing text.")

if (src_001_chunks_df["word_count"] <= 0).any():
    raise ValueError("Some chunks have zero or negative word counts.")

print("SRC-001 chunk validation passed.")
print(f"Validated chunks: {len(src_001_chunks_df)}")
print(f"Total chunk words: {src_001_chunks_df['word_count'].sum()}")
print(f"Average words per chunk: {src_001_chunks_df['word_count'].mean():.1f}")

SRC-001 chunk validation passed.
Validated chunks: 11
Total chunk words: 2729
Average words per chunk: 248.1


In [22]:
# Save SRC-001 processed chunks locally

src_001_chunks_output_path = (
    PROCESSED_DATA_DIR / "src_001_european_commission_ai_act_overview_chunks.csv"
)

src_001_chunks_df.to_csv(src_001_chunks_output_path, index=False, encoding="utf-8")

print(f"Saved SRC-001 chunks to: {src_001_chunks_output_path}")
print(f"File exists: {src_001_chunks_output_path.exists()}")
print(f"File size in bytes: {src_001_chunks_output_path.stat().st_size}")
print(f"Number of saved chunks: {len(src_001_chunks_df)}")

Saved SRC-001 chunks to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\src_001_european_commission_ai_act_overview_chunks.csv
File exists: True
File size in bytes: 20242
Number of saved chunks: 11


In [23]:
# Read saved SRC-001 chunks back to confirm reproducibility

loaded_src_001_chunks_df = pd.read_csv(src_001_chunks_output_path)

print(f"Loaded rows: {len(loaded_src_001_chunks_df)}")
print(f"Loaded columns: {list(loaded_src_001_chunks_df.columns)}")

loaded_src_001_chunks_df.head()

Loaded rows: 11
Loaded columns: ['source_id', 'source_name', 'publisher', 'source_type', 'url', 'planned_raw_filename', 'chunk_id', 'chunk_index', 'chunk_text', 'word_count']


,source_id,source_name,publisher,source_type,url,planned_raw_filename,chunk_id,chunk_index,chunk_text,word_count
0,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_001,1,The AI Act is the first-ever legal framework o...,250
1,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_002,2,"of the AI Act ahead of time. In parallel, the ...",250
2,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_003,3,"prohibits eight practices , namely: harmful AI...",250
3,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_004,4,professional life (e.g. scoring of exams) AI-b...,250
4,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_005,5,"high level of robustness, cybersecurity and ac...",250


In [24]:
# Update source inventory after chunking SRC-001

source_inventory_df.loc[
    source_inventory_df["source_id"] == "SRC-001",
    "processing_status"
] = "chunked"

source_inventory_df.loc[
    source_inventory_df["source_id"] == "SRC-001",
    "processed_chunks_file"
] = src_001_chunks_output_path.name

source_inventory_df.loc[
    source_inventory_df["source_id"] == "SRC-001",
    "chunk_count"
] = len(src_001_chunks_df)

source_inventory_df.to_csv(source_inventory_path, index=False, encoding="utf-8")

source_inventory_df[
    [
        "source_id",
        "source_name",
        "collection_status",
        "processing_status",
        "raw_file_exists",
        "raw_word_count",
        "chunk_count",
        "processed_chunks_file",
    ]
]

,source_id,source_name,collection_status,processing_status,raw_file_exists,raw_word_count,chunk_count,processed_chunks_file
0,SRC-001,AI Act,collected,chunked,True,2229.0,11.0,src_001_european_commission_ai_act_overview_ch...
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,not_collected_yet,not_processed_yet,NaN,NaN,NaN,NaN
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,not_collected_yet,not_processed_yet,NaN,NaN,NaN,NaN
3,SRC-004,Artificial Intelligence Risk Management Framew...,not_collected_yet,not_processed_yet,NaN,NaN,NaN,NaN
4,SRC-005,AI Risk Management Framework overview page,not_collected_yet,not_processed_yet,NaN,NaN,NaN,NaN


## Notebook Summary

This notebook created the first technical foundation for the Responsible AI RAG Assistant project.

Completed steps:

1. Confirmed project folder paths.
2. Defined source metadata for five public AI governance sources.
3. Created a source inventory table.
4. Built reusable text cleaning and chunking helper functions.
5. Tested the chunking pipeline with sample responsible AI text.
6. Saved and reloaded a sample processed chunk file.
7. Built document-loading functions for `.txt` and `.pdf` files.
8. Collected the first real public source: the European Commission AI Act overview page.
9. Saved the raw source text locally.
10. Created processed chunks for `SRC-001`.
11. Saved and reloaded the processed chunk table.
12. Updated the source inventory after successful collection and chunking.

The first real source is now ready for the next notebook, where chunks will be embedded and stored in a vector database.

## Quality Note

The initial web extraction works, but some non-core webpage content may remain in the extracted text.

In later refinement, the project can improve extraction quality by removing navigation text, related links, footer content, and repeated web page boilerplate.

This is a normal part of building a production-style RAG pipeline.

In [25]:
# Final notebook checkpoint

checkpoint_records = [
    {
        "artifact": "sample_responsible_ai_chunks.csv",
        "path": sample_output_path,
        "exists": sample_output_path.exists(),
        "rows": len(loaded_sample_chunks_df),
    },
    {
        "artifact": "source_inventory.csv",
        "path": source_inventory_path,
        "exists": source_inventory_path.exists(),
        "rows": len(source_inventory_df),
    },
    {
        "artifact": "src_001_european_commission_ai_act_overview.txt",
        "path": src_001_output_path,
        "exists": src_001_output_path.exists(),
        "rows": None,
    },
    {
        "artifact": "src_001_european_commission_ai_act_overview_chunks.csv",
        "path": src_001_chunks_output_path,
        "exists": src_001_chunks_output_path.exists(),
        "rows": len(loaded_src_001_chunks_df),
    },
]

checkpoint_df = pd.DataFrame(checkpoint_records)

print("Final notebook checkpoint completed.")
print(f"SRC-001 raw word count: {src_001_loaded['word_count']}")
print(f"SRC-001 chunk count: {len(src_001_chunks_df)}")
print(f"SRC-001 processed chunk file exists: {src_001_chunks_output_path.exists()}")

checkpoint_df

Final notebook checkpoint completed.
SRC-001 raw word count: 2229
SRC-001 chunk count: 11
SRC-001 processed chunk file exists: True


,artifact,path,exists,rows
0,sample_responsible_ai_chunks.csv,c:\Users\mdadg\Desktop\responsible-ai-rag-assi...,True,3.0
1,source_inventory.csv,c:\Users\mdadg\Desktop\responsible-ai-rag-assi...,True,5.0
2,src_001_european_commission_ai_act_overview.txt,c:\Users\mdadg\Desktop\responsible-ai-rag-assi...,True,NaN
3,src_001_european_commission_ai_act_overview_ch...,c:\Users\mdadg\Desktop\responsible-ai-rag-assi...,True,11.0


## Next Notebook

The next notebook will be:

`02_embeddings_and_vector_store.ipynb`

The goal of the next notebook is to:

1. Load processed chunk files.
2. Generate embeddings for the text chunks.
3. Store embeddings in a local Chroma vector database.
4. Test semantic search queries.
5. Retrieve the most relevant chunks for AI governance questions.
6. Prepare the retrieval layer for the RAG assistant.

The next stage converts processed text chunks into searchable vector representations.